# Rhonda Ojongmboh — Playgrounds Sub-metric
### CS 1656 Final Project | Group 4 — Fun Finder Pittsburgh

**Research Question:** What is the most fun neighborhood in Pittsburgh?

**My Sub-metric:** Number of playgrounds per neighborhood

**Dataset:** [City of Pittsburgh Playgrounds — WPRDC](https://data.wprdc.org/dataset/playgrounds)

**Team Members:** Sachit Anand, Rhonda Ojongmboh, Osarumen Okhiku

---
## 1. Introduction

Playgrounds are one of the most direct measures of a neighborhood's commitment to fun. Unlike parks — which can be large grassy fields or passive green spaces — playgrounds are purpose-built for active play. They have equipment, they are designed for people to use them energetically, and they serve as community gathering points for families.

My contribution to the group's "most fun" metric is the number of city-maintained playgrounds per neighborhood. The more playgrounds a neighborhood has, the more opportunities residents have to be active and playful, which directly maps to our definition of fun.

In this notebook I will:
1. Load and explore the City of Pittsburgh Playgrounds dataset
2. Count the number of playgrounds in each neighborhood
3. Visualize the results with bar and pie charts
4. Identify the top neighborhood by playground count
5. Export a normalized score for use in the group combined metric

---
## 2. Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import os

pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

os.makedirs('data', exist_ok=True)
print('Libraries loaded successfully.')

---
## 3. Load the Data

In [ ]:
# Load from local CSV (place playgrounds.csv inside a data/ folder next to this notebook)
playgrounds = pd.read_csv('data/playgrounds.csv')

print(f'Rows: {len(playgrounds)}')
print(f'Columns: {list(playgrounds.columns)}')
playgrounds.head()

---
## 4. Explore the Data

In [ ]:
print(f'Total playgrounds in the dataset: {len(playgrounds)}')
print(f'Unique neighborhoods with at least one playground: {playgrounds["neighborhood"].nunique()}')
print(f'\nMissing values per column:')
print(playgrounds.isnull().sum())

In [ ]:
# Which parks are these playgrounds located in?
print('Top 10 parks that contain playgrounds:')
print(playgrounds['park'].value_counts().head(10))

print('\nWho maintains the playgrounds?')
print(playgrounds['maintenance_responsibility'].value_counts())

In [ ]:
# Quick look at neighborhood distribution
print('Sample of neighborhood values:')
print(playgrounds['neighborhood'].value_counts().head(10))

---
## 5. Count Playgrounds per Neighborhood

In [ ]:
# Remove any rows where neighborhood is missing
pg_clean = playgrounds.dropna(subset=['neighborhood']).copy()
print(f'Rows after dropping missing neighborhoods: {len(pg_clean)}')

# Count playgrounds per neighborhood and sort descending
pg_per_hood = (
    pg_clean
    .groupby('neighborhood')
    .size()
    .reset_index(name='playground_count')
    .sort_values('playground_count', ascending=False)
    .reset_index(drop=True)
)

print(f'\nNeighborhoods ranked by playground count (top 15):')
print(pg_per_hood.head(15).to_string(index=False))

---
## 6. Visualizations

In [ ]:
# ── Chart 1: Top 15 neighborhoods by playground count ────────────────────────
top15 = pg_per_hood.head(15)

fig, ax = plt.subplots(figsize=(14, 7))

colors = ['#e74c3c' if i == 0 else '#3498db' for i in range(len(top15))]
bars = ax.bar(top15['neighborhood'], top15['playground_count'],
              color=colors, edgecolor='white', linewidth=0.8)

for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            str(int(bar.get_height())), ha='center', va='bottom',
            fontsize=10, fontweight='bold')

ax.set_title('Top 15 Pittsburgh Neighborhoods by Number of Playgrounds',
             fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Neighborhood', fontsize=12)
ax.set_ylabel('Number of Playgrounds', fontsize=12)
ax.set_xticklabels(top15['neighborhood'], rotation=45, ha='right', fontsize=10)
ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.legend(handles=[
    plt.Rectangle((0,0),1,1, color='#e74c3c', label='#1 Neighborhood'),
    plt.Rectangle((0,0),1,1, color='#3498db', label='Other Top 15')
], fontsize=10)

plt.tight_layout()
plt.savefig('playgrounds_top15.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: playgrounds_top15.png')

In [ ]:
# ── Chart 2: All neighborhoods ranked (horizontal) ───────────────────────────
sorted_all = pg_per_hood.sort_values('playground_count')

fig, ax = plt.subplots(figsize=(10, 18))
ax.barh(sorted_all['neighborhood'], sorted_all['playground_count'],
        color='#3498db', edgecolor='white')

ax.set_title('All Pittsburgh Neighborhoods Ranked by Playground Count',
             fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Number of Playgrounds', fontsize=12)
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig('playgrounds_all_neighborhoods.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: playgrounds_all_neighborhoods.png')

In [ ]:
# ── Chart 3: Pie — top 5 vs. rest of Pittsburgh ───────────────────────────────
top5       = pg_per_hood.head(5)
rest_count = pg_per_hood.iloc[5:]['playground_count'].sum()

labels  = list(top5['neighborhood']) + ['All Other Neighborhoods']
sizes   = list(top5['playground_count']) + [rest_count]
colors  = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6','#bdc3c7']
explode = [0.06] + [0] * 5

fig, ax = plt.subplots(figsize=(9, 9))
ax.pie(sizes, labels=labels, colors=colors, explode=explode,
       autopct='%1.1f%%', startangle=140, textprops={'fontsize': 11})
ax.set_title('Share of Playgrounds: Top 5 Neighborhoods vs. Rest of Pittsburgh',
             fontsize=13, fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig('playgrounds_pie.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: playgrounds_pie.png')

---
## 7. Results

In [ ]:
winner = pg_per_hood.iloc[0]
total_pg = pg_per_hood['playground_count'].sum()

print('=' * 54)
print(f'  Playgrounds Sub-metric Winner: {winner["neighborhood"]}')
print(f'  Playground count:   {winner["playground_count"]}')
print(f'  % of all city playgrounds: {winner["playground_count"]/total_pg*100:.1f}%')
print('=' * 54)
print()
print('Full top 5:')
print(pg_per_hood.head(5).to_string(index=False))

**Squirrel Hill South** leads with 8 playgrounds — 3 more than the second-place Beechview and South Side Slopes (5 each). Squirrel Hill South has 6.4% of all city playgrounds, making it the clear leader in this sub-metric.

---
## 8. Normalized Score for Group Combination

To combine fairly with Sachit's parks score and Osarumen's courts/rinks score, the playground count is normalized to a 0–1 scale where 1.0 = the neighborhood with the most playgrounds (Squirrel Hill South, 8).

In [ ]:
pg_per_hood['playground_score'] = (
    pg_per_hood['playground_count'] / pg_per_hood['playground_count'].max()
)

# Export for use in the group notebook
pg_per_hood[['neighborhood', 'playground_count', 'playground_score']].to_csv(
    'data/playgrounds_scores.csv', index=False
)

print('Exported: data/playgrounds_scores.csv')
print()
print('Top 10 with normalized scores:')
print(pg_per_hood[['neighborhood', 'playground_count', 'playground_score']].head(10).to_string(index=False))

---
## 9. Conclusion

Based on the playgrounds sub-metric alone, **Squirrel Hill South** is the most fun neighborhood in Pittsburgh with 8 city playgrounds. This is a clear lead over every other neighborhood, and it reflects Squirrel Hill South's strong residential character and family-oriented community infrastructure.

Playgrounds are a particularly meaningful measure of fun because they are explicitly designed for play — they exist for no other reason than recreation. A neighborhood investing in multiple playgrounds is one that prioritizes giving residents, especially families and children, places to be active and social.

**Personal reflection:** Squirrel Hill South as the playground leader actually matches my own sense of the neighborhood — it has always felt like a very community-oriented, livable area. Personally, I find neighborhoods like the South Side more fun because of the social scene, but from a pure recreational infrastructure standpoint, the data is convincing. Squirrel Hill South not only tops the playground count but also comes out as the overall winner when we combine all three sub-metrics, which tells me it is genuinely well-equipped across the board — not just a one-category winner.